# COMP8410 Advanced Topic in Artificial Intelligence

# Notebook 3 - Global Pricing (Which price tier fits each new K-pop merchandise item?)

> this will help to check if price can match which market price and not over price.

### Introduction

When a K-pop merchandise launches new comeback merchandise with Artist' single in Korean and US online stores, it must decide which **price tier**  fits each item, and spot listings that look overpriced compared with similar products.



This notebook compares two graph neural networks (GNNs) that predict the price tier of each product from its features **and** from the products it is linked to.

| | System A - Standard System Overview | System B - Global System Overview | Baseline |
|---|---|---|---|
| Model | **GAT** | **FAGCN**  | **MLP** with no graph |
| Idea | attention over neighbours; weights are always positive, so it acts as a **low-pass filter** (neighbours are assumed similar) | self-gating mixes **low- and high-frequency** signals with signed weights, so it can also learn from neighbours that are different | uses only the item's own features |

### Linked with COMP8410 Knowledge
* Week 11 - GNNs under homophily and heterophily
Furthermore, this experiment will test and check through
  * **Same-artist edges:** a sticker and a lightstick of one group usually sit in different price tiers.
  * **Same-category edges:** products of the same type usually cost about the same 

### Dataset
**KPop Online Merchandise Prices** (Wan, 2022), a public Kaggle dataset and No token is needed - https://www.kaggle.com/datasets/ericwan1/kpop-merchandise-prices


**References**
- Bo, D., Wang, X., Shi, C., & Shen, H. (2021). Beyond low-frequency information in graph convolutional networks. *Proceedings of the AAAI Conference on Artificial Intelligence, 35*(5), 3950–3957. https://multisearch.mq.edu.au/permalink/61MACQUARIE_INST/1gnm2jp/cdi_crossref_citationtrail_10_1609_aaai_v35i5_16514
- Zhu, J., Yan, Y., Zhao, L., Heimann, M., Akoglu, L., & Koutra, D. (2020). Beyond homophily in graph neural networks: Current limitations and effective designs. *Advances in Neural Information Processing Systems (NeurIPS), 33*, 7793–7804. https://multisearch.mq.edu.au/permalink/61MACQUARIE_INST/1gnm2jp/cdi_arxiv_primary_2006_11468

# 1. Import Libraries and Set-up

In [ ]:
# Shared settings: identical in every notebook (if you change one, change all) 
from pathlib import Path
import os, sys, json, time, random, platform, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


# Experiment Settings for seed
SEED = 42


SYSTEMS = {"A": "Standard System Overview", "B": "Global System Overview"}

**GPU Checking**
* Cost estimation for computing unit and GPU type
* All of the price references and Google Colab Information in **ENV_SETUP.ipynb**

In [ ]:
COLAB_PRICE_PER_UNIT_USD = 9.99 / 100

COLAB_UNITS_PER_HOUR_BY_GPU = {"T4": 1.19, "L4": 1.71, "A100": 5.40}


# GPU Detection
try:

    import torch

    _gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

except ImportError:

    _gpu = "CPU"


# Compute the approximate GPU cost per hour in USD
GPU_HOURLY_USD = next(
    (u for k, u in COLAB_UNITS_PER_HOUR_BY_GPU.items() if k in _gpu), 0.0 ) * COLAB_PRICE_PER_UNIT_USD

In [ ]:
#  Random Seed Function 
def set_seed(seed: int = SEED) -> None:

    
    random.seed(seed)

    np.random.seed(seed)

    try:
        import torch

        torch.manual_seed(seed)

        torch.cuda.manual_seed_all(seed)

    except ImportError:
        
        pass

In [ ]:
#  Device Detection Function 


def get_device() -> str:

    try:
        import torch

        if torch.cuda.is_available():

            return "cuda"

    except ImportError:
        pass

    return "cpu"



set_seed()


print(
    f"device: {get_device()} | "
    f"GPU: {_gpu} ≈ US${GPU_HOURLY_USD:.3f}/hour"
)

**Helper Function**

This will help me to check about run metadata (Dataset), timing, outcomes, which will use in all pairs model

So there is mainly feature:
* Check GPU 
* Model management
* Saved program environment
* GPU memory setuo
* Timimg measure
* Save final experiment result

In [ ]:
# GPU Type and Capability
def gpu_dtype():

    import torch

    if torch.cuda.is_available():

        if torch.cuda.get_device_capability()[0] >= 8:
            return torch.bfloat16
        else:
            return torch.float16

    return torch.float32

In [ ]:
# Transformers Output from AI model to Tensor Function - Model Management
def as_tensor(out):

    """get_*_features returns a tensor in transformers 4.x; newer versions may wrap it in an output object."""

    if hasattr(out, "pooler_output") and out.pooler_output is not None:

        return out.pooler_output

    if hasattr(out, "image_embeds"):
        
        return out.image_embeds

    if hasattr(out, "text_embeds"):

        return out.text_embeds

    return out if hasattr(out, "shape") else out[0]

In [ ]:
# Store Environment Information Function 

def env_info():

    """Saved inside every result file as evidence of when and where it was produced."""


    info = {

        "timestamp_utc": pd.Timestamp.now(tz="UTC").isoformat(timespec="seconds"),

        "python": platform.python_version(),

        "os": platform.platform()

    }

    for pkg in ("torch", "transformers", "diffusers", "torch_geometric"):

        try:

            info[pkg] = __import__(pkg).__version__

        except Exception:

            info[pkg] = None

    try:

        import torch

        info["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None


    except Exception:

        info["gpu"] = None

    return info

In [ ]:
# Logging Function for GPU Synchronization for Accurate Timing and Checking GPU working

def sync_gpu():

    try:

        import torch

        if torch.cuda.is_available():

            torch.cuda.synchronize()

    except ImportError:

        pass

In [ ]:
# Free GPU Memory Function to Avoid Out of Memory Errors

def free_gpu():

    import gc

    gc.collect()

    try:

        import torch

        torch.cuda.empty_cache()

    except ImportError:

        pass

In [ ]:

# Timer -> for measuring latency and peak GPU memory usage, leading to performance evaluation and cost analysis of AI models

class Timer:
    #  check model performance from Latency and Peak VRAM


    def __init__(self):
        # Constructor will initialize the Timer object and set default values for latencies and peak VRAM usage.

        self.latencies, self.peak_vram_gb = [], None

        try:
           # start checking GPU
            import torch

            if torch.cuda.is_available():
    
                torch.cuda.reset_peak_memory_stats()

        except ImportError:

            pass




    def add(self, seconds, n_items=1):
        # this will calculate the average time per item and store it in the latencies list for further analysis.

        self.latencies.extend([seconds / n_items] * n_items)


    def finish(self):
        # this function is called after all processing is done to finalize the timing and memory usage data.
        try:

         # check GPU again
            import torch

            if torch.cuda.is_available():

                torch.cuda.synchronize()


                self.peak_vram_gb = torch.cuda.max_memory_allocated() / 1024**3
               # check GPU Memory usage and convert to GB

        except ImportError:

            pass

        return self



    def business(self):

        # this function summarizes the business performance metrics of the AI model to provide insights for business decision-making and cost analysis.

        mean = float(np.mean(self.latencies)) if self.latencies else float("nan")
        # Compute the average latency in seconds for processing each item, and handle cases where there are no latencies recorded by returning NaN.

        return {

            "mean_latency_s": round(mean, 4),


            "p95_latency_s": round(float(np.percentile(self.latencies, 95)), 4) if self.latencies else None,
             # 95th percentile latency, indicating that 95% of the items were processed in this time or less. If there are no latencies recorded, it returns None.

            "gpu_hourly_usd": round(GPU_HOURLY_USD, 4),

            "cost_per_1000_usd": round(mean * 1000 / 3600 * GPU_HOURLY_USD, 4),

            "peak_vram_gb": None if self.peak_vram_gb is None else round(self.peak_vram_gb, 2),
          
        }

In [ ]:
# JSON file handling function to manage data that cannot be directly converted to JSON, such as NumPy Scalars or special object types.
def _json_default(o):
    

    return o.item() if hasattr(o, "item") else str(o)

In [ ]:
# Save Result Function

# Project folders: relative paths work on both Colab and Windows
ROOT = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
RAW = ROOT / "Datasets"
PROCESSED = RAW / "processed"
AMAZON_DIR = RAW / "Amazon2023_Merchandise_Data"
RESULTS = ROOT / "Results"
FIGURES, TABLES, METRICS = RESULTS / "figures", RESULTS / "tables", RESULTS / "metrics"
for _d in (PROCESSED, AMAZON_DIR, FIGURES, TABLES, METRICS):
    _d.mkdir(parents=True, exist_ok=True)


def save_result(task, system, model_id, technical, business,
                governance, notes="", items=None, extra=None):

    records = {

        "task": task,

        "system": system,

        "model_id": model_id,

        "technical": technical,

        "business": business,

        "governance": governance,

        "notes": notes,

        "env": env_info(),

        "extra": extra or {}

    }

    path = METRICS / f"{task}__{system}.json"

    path.write_text(
        json.dumps(
            records,
            indent=2,
            ensure_ascii=False,
            default=_json_default
        ),
        encoding="utf-8"
    )

    if items is not None:

        items.to_csv(
            TABLES / f"{task}__{system}_items.csv",
            index=False
        )

    print("saved", path)

    return records

## 3. Task Settings

For this task, I compares three models to see how well they can predict K-pop merchandise price categories. System A uses GAT, System B uses FAGCN, and the baseline is a simple MLP that does not use graph connections. All three models predict whether a product belongs to the **budget, mid, or premium** price category.

### Data
- I only use `all_shops.csv` from the KPop Online Merchandise Prices dataset because it already includes data from all six shops. Using the other six files as well would duplicate the products. this is form my survey
- I divide the products into three roughly equal-sized price groups: **budget, mid, and premium**. These are the categories the models try to predict.
- I do not include the actual price as an input because that would give the models the answer. Instead, they use other product information, such as the product name, store, region, whether the item is sold out, and whether it is autographed.

In [ ]:
TASK = "3_Global_Pricing"
DATA_DIR = RAW / "KPop Online Merchandise Prices"  
DATA_FILE = "all_shops.csv"           # combines the other 6 shop files -> read only this one to avoid duplicates
COLUMN_MAP = {}                       # override auto-detection,
CURRENCY_TO_USD = {"USD": 1.0, "KRW": 0.00073, "JPY": 0.0067}   # come with cite and source date
N_TIERS = 3                           # price ranking
MAX_NEIGHBOURS = 10                   
EDGE_TYPES = ["artist", "category"]   
PRIMARY_GRAPH = "artist"              # the graph used for the A vs B decision -> real bu growth
MODELS = {
    "A": {"kind": "gat", "hidden": 64, "heads": 4, "dropout": 0.5},
    "B": {"kind": "fagcn", "hidden": 64, "layers": 2, "eps": 0.3, "dropout": 0.5},
    "baseline_mlp": {"kind": "mlp", "hidden": 64, "dropout": 0.5},
}
TRAIN = {"epochs": 300, "lr": 0.005, "weight_decay": 5e-4, "patience": 50}
SPLIT = {"train": 0.6, "val": 0.2, "test": 0.2}
N_SEEDS = 5                           # repeat every model on 5 random splits -> mean and std

# 4. Load data and Clean it

Column names are detected automatically (name, price, artist, store, category). 

In [ ]:
import re

#  column names in different datasets
CANDIDATES = {
    "name": ["name", "title", "item", "product", "product_name", "item_name"],
    "price": ["price", "price_usd", "cost", "amount", "sale_price"],
    "artist": ["artist", "group", "idol", "band", "artist_name"],
    "vendor": ["vendor", "store", "shop", "site", "source", "seller", "website"],
    "category": ["category", "type", "product_type", "item_type"],
    "currency": ["currency"],
}

# Keywords to tackle product types
CATEGORY_KEYWORDS = {
    "lightstick": ["lightstick", "light stick", "fanlight"],
    "photobook": ["photobook", "photo book", "photo-book"],
    "photocard": ["photocard", "photo card", "pc set"],
    "album": ["album", "mini", "repackage", "ep ", "cd"],
    "poster": ["poster"], "sticker": ["sticker"],
    "keyring": ["keyring", "key ring", "keychain"],
    "hoodie": ["hoodie", "sweatshirt"], "t-shirt": ["t-shirt", "tee", "tshirt"],
    "bag": ["bag", "pouch", "tote"], "season's greetings": ["season", "calendar", "greetings"],
}

# Store names used to guess which shown entertainment industry or k-pop community chaneel
KR_VENDORS = ["mwave", "sm", "jyp", "ktown", "weverse"]
US_VENDORS = ["musicplaza", "kpopalbums", "kpopstoreinusa", "usa"]


# Find the correct column even if datasets use different names
def find_column(cols, keys):
    low = {c.lower().strip(): c for c in cols}
    for k in keys:
        if k in low:
            return low[k]
    for k in keys:
        for lc, c in low.items():
            if k in lc:
                return c
    return None


# Convert prices into numbers
def convert_price(v):
    if pd.isna(v):
        return np.nan
    if isinstance(v, (int, float)):
        return float(v)
    s = re.sub(r"[^\d.,]", "", str(v))
    if "," in s and "." not in s:
        s = s.replace(",", "") if len(s.split(",")[-1]) == 3 else s.replace(",", ".")
    try:
        return float(s.replace(",", ""))
    except ValueError:
        return np.nan


# Guess the product category from its name
def refer_category(name):
    n = f" {str(name).lower()} "
    for cat, kws in CATEGORY_KEYWORDS.items():
        if any(k in n for k in kws):
            return cat
    return "other"


# Guess whether a store is in Korea or the US
def refer_region(vendor):
    v = str(vendor).lower().replace(" ", "")
    if any(k in v for k in US_VENDORS):
        return "US"
    if any(k in v for k in KR_VENDORS):
        return "KR"
    return "other"


# Get the artist name from the beginning of the product name such TWICE wiht Album name or BTS with ALBUM name
def artist_from_name(names):
    return names.str.extract(r"^\s*\[?([A-Za-z0-9&.\- ]{2,20}?)[\]\s\-:]", expand=False).fillna("unknown")


# Clean the raw data and prepare prices in USD
def USD_prices_prepare(raw):
    cols = {k: COLUMN_MAP.get(k) or find_column(raw.columns, keys) for k, keys in CANDIDATES.items()}
    if cols["price"] is None or cols["name"] is None:
        raise ValueError(f"Could not detect name/price in {list(raw.columns)}: set COLUMN_MAP")

    out = pd.DataFrame({"name": raw[cols["name"]].astype(str)})
    out["price_raw"] = raw[cols["price"]].map(convert_price)
    out["vendor"] = raw[cols["vendor"]].astype(str) if cols["vendor"] else raw.get("_source_file", "unknown")
    out["category"] = raw[cols["category"]].astype(str).str.lower() if cols["category"] else out["name"].map(refer_category)

    # Use the artist column when available OR guess from the product name
    from_name = artist_from_name(out["name"])
    if cols["artist"]:
        given = raw[cols["artist"]].where(raw[cols["artist"]].notna() & (raw[cols["artist"]].astype(str).str.strip() != ""))
        out["artist"] = given.astype("object").fillna(from_name).astype(str)
    else:
        out["artist"] = from_name

    # Keep Artist name tagged
    out["artist_source"] = np.where(cols["artist"] and raw[cols["artist"]].notna(), "column", "inferred") if cols["artist"] else "inferred"

    out["region"] = out["vendor"].map(refer_region)

    # If currency is missing -> use store price to guess KRW or USD
    if cols["currency"]:
        cur = raw[cols["currency"]].astype(str).str.upper().str.strip()
    else:
        cur = pd.Series(np.where(out.groupby("vendor")["price_raw"].transform("median") > 1000, "KRW", "USD"), index=out.index)

    # Convert all prices to USD
    out["currency"] = cur
    out["price_usd"] = out["price_raw"] * cur.map(CURRENCY_TO_USD).fillna(1.0)

    # Remove missing prices, invalid prices, and duplicate products
    out = out.dropna(subset=["price_usd"])
    out = out[out["price_usd"] > 0].drop_duplicates(subset=["name", "vendor"]).reset_index(drop=True)
    out.attrs["detected_columns"] = cols
    return out


# Create sample data products with prices based mainly on product category.
def synthetic_prices(n_artists=40, items_per_artist=25, seed=SEED):
    rng = np.random.default_rng(seed)
    cats = {"sticker": (6, .3), "keyring": (12, .3), "photocard": (8, .4), "poster": (15, .3), "album": (22, .25),
            "t-shirt": (35, .2), "hoodie": (60, .2), "photobook": (45, .3), "lightstick": (55, .15)}
    vendors = {"Official KR Shop": "KR", "Seoul Music Mall": "KR", "US Kpop Store": "US", "Kpop Albums USA": "US"}
    rows = []
    for a in range(n_artists):
        pop = rng.lognormal(0, .25)
        for _ in range(items_per_artist):
            cat, vendor = rng.choice(list(cats)), rng.choice(list(vendors))
            med, spread = cats[cat]
            price = med * pop * (1.15 if vendors[vendor] == "US" else 1.0) * rng.lognormal(0, spread)
            rows.append({"name": f"Artist{a:02d} {cat} ver.{rng.integers(1, 4)}", "artist": f"Artist{a:02d}",
                         "category": cat, "vendor": vendor, "region": vendors[vendor], "price_usd": round(float(price), 2)})
    return pd.DataFrame(rows).drop_duplicates(subset=["name", "vendor"]).reset_index(drop=True)


# Load CSV file
def load_prices():
    if USE_SYNTHETIC:
        return synthetic_prices()

    
    path = DATA_DIR / DATA_FILE

    # Check that the dataset is available 
    if not path.exists():
        raise FileNotFoundError(f"Put {DATA_FILE} in {DATA_DIR} (or set USE_SYNTHETIC = True)")

    raw = pd.read_csv(path).assign(_source_file=path.stem)
    return USD_prices_prepare(raw)


# Load the cleaned dataset
df = load_prices()

# Show the detected columns and artist information
print("detected columns:", df.attrs.get("detected_columns", "synthetic"))
if "artist_source" in df:
    print("artist source:", df["artist_source"].value_counts().to_dict())

# Save the cleaned data for later use
df.to_csv(PROCESSED / "kpop_prices_clean.csv", index=False)

# Show a quick summary of the dataset
display(pd.DataFrame({"rows": [len(df)], "artists": [df["artist"].nunique()], "stores": [df["vendor"].nunique()],
                      "categories": [df["category"].nunique()], "median_price_usd": [round(df["price_usd"].median(), 2)],
                      "regions": [df["region"].value_counts().to_dict()]}))

# Show 8 random products to check the data
df.sample(8, random_state=0)

**Pull Cleaned Data from Google Drive First**

As I directly collect data in google drive due to memory unit in local laptop not enough to user and laptop have crashed before too.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil
from pathlib import Path

# Save the finished CSV file to Google Drive
source = PROCESSED / "kpop_prices_clean.csv"
destination = Path("/content/drive/MyDrive/kpop_prices_clean.csv")

shutil.copy2(source, destination)

print("Saved to Google Drive:", destination)